# MetroDemand Analytics
Diseño e implementación de una solución Big Data para el análisis y predicción de la demanda horaria del Metro de Medellín.

Procesamiento PySpark, persistencia Delta Lake y modelación Spark MLlib. El lector Excel es auxiliar. Horarios oficiales del Metro de Medellín suministrados para el proyecto. Ejecutar en Databricks con acceso al volumen de entrada y permisos sobre los esquemas de salida. Configurar las rutas en la sección 1.

## 0. Configuración del entorno

In [ ]:
%pip install openpyxl holidays matplotlib

In [ ]:
from datetime import time, date
import os
from uuid import uuid4
import pandas as pd
import holidays
import mlflow
import mlflow.spark
from pyspark.sql import functions as F, Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor


CATALOGO = "workspace"
ESQUEMA_FUENTE = "default"
VOLUMEN = "data"
RUTA_DATA = f"/Volumes/{CATALOGO}/{ESQUEMA_FUENTE}/{VOLUMEN}/"
PERMITIR_SOBRESCRITURA = True
USUARIO = spark.sql("SELECT current_user() AS usuario").first().usuario
EXPERIMENTO_MLFLOW = f"/Users/{USUARIO}/MetroDemand_Analytics"
ID_EJECUCION = uuid4().hex
os.environ["MLFLOW_DFS_TMP"] = RUTA_DATA.rstrip("/")

def tabla_analitica(nombre):
    return f"{CATALOGO}.analitica.{nombre}"
LINEAS_OBJETIVO = ["LÍNEA A", "LÍNEA B", "LÍNEA 1", "LÍNEA 2", "LÍNEA T-A"]
HORAS = [f"h_{h:02d}" for h in range(4, 24)]
CLAVES = ["fecha", "linea", "hora_num"]
TABLA_BRONZE = f"{CATALOGO}.bronze.metrodemanda_afluencia_raw"
TABLA_SILVER = f"{CATALOGO}.silver.metrodemanda_afluencia"
TABLA_GOLD = f"{CATALOGO}.gold.metrodemanda_afluencia_horaria_base"
TABLA_GOLD_COMPLETA = f"{CATALOGO}.gold.metrodemanda_afluencia_horaria_completa"
TABLA_PREDICCIONES = f"{CATALOGO}.analitica.metrodemanda_predicciones_ago_dic2026"
for esquema in ["bronze", "silver", "gold", "analitica"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{esquema}")

def guardar(df, tabla):
    if not PERMITIR_SOBRESCRITURA and spark.catalog.tableExists(tabla):
        raise RuntimeError(f"La tabla ya existe: {tabla}")
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla)
    return spark.table(tabla)

def verificar(df, claves):
    assert df.filter(F.expr(" OR ".join(f"`{c}` IS NULL" for c in claves))).count() == 0
    assert df.groupBy(*claves).count().filter("count > 1").count() == 0

print("Spark:", spark.version)
print("Dependencias:", {"pandas":pd.__version__,"holidays":holidays.__version__,"mlflow":mlflow.__version__})
import matplotlib.pyplot as plt
from datetime import timedelta
from itertools import product
RF_GRID = list(product([50,100],[5,10,15],[5,10]))
RF_PARAMS = (50,10,5)


## 1. Bronze — Ingesta y calidad

In [ ]:
def leer_excel(anio, hoja, esperado):
    archivo = f"Afluencia_Metro_{anio}.xlsx"
    datos = pd.read_excel(RUTA_DATA + archivo, sheet_name=hoja, header=1)
    datos.insert(0, "fila_excel_origen", range(3, len(datos)+3))
    datos = datos[datos["Unnamed: 0"].astype(str).str.strip() != "Resultado total"].copy()
    assert len(datos) == esperado, f"Cantidad inesperada en {archivo}"
    datos = datos.rename(columns={c: f"h_{c.hour:02d}" for c in datos.columns if isinstance(c, time)})
    datos = datos.rename(columns={"Unnamed: 0":"fecha", "Unnamed: 1":"linea", "Total general (Número de pasajeros)":"total_general"})
    datos["fecha"] = pd.to_datetime(datos["fecha"], errors="raise").dt.strftime("%Y-%m-%d")
    datos["linea"] = datos["linea"].astype(str).str.strip()
    datos["archivo_fuente"] = archivo
    datos["anio_fuente"] = anio
    columnas = ["fecha", "linea", "archivo_fuente", "anio_fuente", "fila_excel_origen", "total_general", *HORAS]
    assert not datos.columns.duplicated().any()
    for c in ["total_general", *HORAS]:
        datos[c] = pd.to_numeric(datos[c], errors="raise").astype(float)
    return datos[columnas]

entrada = pd.concat([leer_excel(2025,"Hoja1",3214), leer_excel(2026,"Afluencia_Metro_2026",2507)], ignore_index=True)
bronze = spark.createDataFrame(entrada)
del entrada
for c in ["total_general", *HORAS]:
    bronze = bronze.withColumn(c, F.when(F.isnan(c), F.lit(None).cast("double")).otherwise(F.col(c)))
bronze = bronze.withColumn("fecha", F.to_date("fecha")).withColumn("fecha_ingesta", F.current_timestamp())
assert bronze.count() == 5721
assert bronze.filter(F.year("fecha") != F.col("anio_fuente")).count() == 0
verificar(bronze, ["archivo_fuente", "fila_excel_origen"])
bronze = guardar(bronze, TABLA_BRONZE)
display(bronze.groupBy("anio_fuente").count())

## 2. Silver — Normalización y calidad

In [ ]:
origen = bronze.filter(F.col("linea").isin(LINEAS_OBJETIVO))
expresion = "stack(20, " + ", ".join(f"'{h}', `{h}`" for h in HORAS) + ") as (hora, demanda)"
silver = origen.selectExpr("fecha", "linea", "total_general", "archivo_fuente", "fila_excel_origen", expresion).withColumn("hora_num", F.substring("hora",3,2).cast("int"))
verificar(silver, CLAVES)
assert silver.count() == origen.count()*20
assert silver.filter("demanda < 0").count() == 0
control = silver.groupBy("fecha","linea").agg(F.sum("demanda").alias("suma_horas"), F.first("total_general").alias("total_origen"))
assert control.filter(F.abs(F.col("suma_horas")-F.col("total_origen")) > 0.01).count() == 0
silver = guardar(silver, TABLA_SILVER)
display(silver.groupBy("linea").agg(F.count("*").alias("registros"), F.count("demanda").alias("observados")))

## 3. Gold — Enriquecimiento analítico
Horarios oficiales suministrados; franjas interpretadas como intervalos [hora, hora+1).

In [ ]:
festivos = sorted(holidays.country_holidays("CO", years=[2024,2025,2026,2027]).keys())
df_festivos = spark.createDataFrame([(d,) for d in festivos], "festivo date")

def calendario(df):
    for nombre, desplazamiento in [("es_festivo",0),("es_dia_previo_festivo",1),("es_dia_posterior_festivo",-1)]:
        fechas = df_festivos.select(F.date_add("festivo",-desplazamiento).alias("fecha"), F.lit(1).alias(nombre))
        df = df.join(F.broadcast(fechas), "fecha", "left").fillna(0, subset=[nombre])
    df = df.withColumn("anio",F.year("fecha")).withColumn("mes",F.month("fecha")).withColumn("dia",F.dayofmonth("fecha")).withColumn("dia_semana_num",F.dayofweek("fecha"))
    especial = (F.col("dia_semana_num")==1) | (F.col("es_festivo")==1)
    return (df.withColumn("tipo_dia", F.when(F.col("es_festivo")==1,"Festivo").when(F.col("dia_semana_num")==1,"Domingo").when(F.col("dia_semana_num")==7,"Sábado").otherwise("Día hábil"))
        .withColumn("fin_de_semana",F.col("dia_semana_num").isin(1,7).cast("int"))
        .withColumn("en_horario_operacion",F.when(especial,F.col("hora_num").between(5,21)).otherwise(F.col("hora_num").between(4,22)).cast("int"))
        .withColumn("operacion_parcial",((~especial)&F.col("hora_num").isin(4)).cast("int"))
        .withColumn("hora",F.format_string("h_%02d",F.col("hora_num"))))

def estructura(inicio, fin):
    fechas = spark.sql(f"SELECT explode(sequence(date'{inicio}', date'{fin}', interval 1 day)) AS fecha")
    lineas = spark.createDataFrame([(l,) for l in LINEAS_OBJETIVO],"linea string")
    horas = spark.range(4,24).select(F.col("id").cast("int").alias("hora_num"))
    return calendario(fechas.crossJoin(lineas).crossJoin(horas))

gold = guardar(calendario(silver), TABLA_GOLD)
assert gold.filter(~F.col("fecha").between("2025-01-01","2026-07-31")).count()==0
reales = gold.filter("en_horario_operacion = 1 AND demanda IS NOT NULL")

verificar(gold,CLAVES)


In [ ]:
base = estructura("2025-01-01","2026-07-31").join(gold.select(*CLAVES,F.col("demanda").alias("demanda_original"),F.col("total_general").alias("total_general_origen"),"archivo_fuente","fila_excel_origen",F.lit(1).alias("fila_presente")),CLAVES,"left")
base = base.withColumn("estado_dato_original",F.when(F.col("demanda_original").isNotNull(),"OBSERVADO").when(F.col("fila_presente").isNull(),"FECHA_HORA_AUSENTE").otherwise("DEMANDA_NULA"))

verificar(base,CLAVES)
assert base.count()==57700


## 4. Análisis exploratorio

In [ ]:
def grafica_lineas(df, x, y, titulo):
    p = df.toPandas()
    fig, ax = plt.subplots(figsize=(11,4))
    for linea, g in p.groupby('linea'):
        g = g.sort_values(x); ax.plot(g[x],g[y],label=linea)
    ax.set(title=titulo,ylabel='Pasajeros'); ax.legend(); fig.autofmt_xdate(); plt.show(); plt.close(fig)

def perfil(df, valor, titulo):
    grafica_lineas(df.groupBy('linea','hora_num').agg(F.avg(valor).alias('promedio')), 'hora_num','promedio',titulo)

def calor(df, valor, titulo):
    p=df.groupBy('linea','dia_semana_num','hora_num').agg(F.avg(valor).alias('v')).toPandas()
    for linea,g in p.groupby('linea'):
        matriz=g.pivot(index='dia_semana_num',columns='hora_num',values='v').reindex(index=range(1,8),columns=range(4,24))
        fig,ax=plt.subplots(figsize=(10,3)); im=ax.imshow(matriz,aspect='auto'); ax.set(title=f'{titulo} — {linea}',xticks=range(20),xticklabels=range(4,24),yticks=range(7),yticklabels=['Dom','Lun','Mar','Mié','Jue','Vie','Sáb']); fig.colorbar(im,ax=ax); plt.show(); plt.close(fig)

display(reales.groupBy('linea').agg(F.count('demanda').alias('n'),F.avg('demanda').alias('media'),F.stddev('demanda').alias('desviacion'),F.min('demanda').alias('minimo'),F.max('demanda').alias('maximo')))
perfil(reales,'demanda','Perfil horario observado')
calor(reales,'demanda','Demanda observada')
grafica_lineas(reales.groupBy('linea','fecha').agg(F.sum('demanda').alias('total')), 'fecha','total','Demanda diaria observada (cobertura variable)')
grafica_lineas(reales.groupBy('linea',F.trunc('fecha','month').alias('mes_fecha')).agg(F.sum('demanda').alias('total')), 'mes_fecha','total','Demanda mensual observada (cobertura variable)')
p=reales.select('linea','demanda').toPandas(); p.boxplot(column='demanda',by='linea',figsize=(10,4)); plt.suptitle(''); plt.show(); plt.close()
display(reales.groupBy("linea","hora_num").agg(F.avg("demanda").alias("promedio"),F.stddev("demanda").alias("desviacion"),F.max("demanda").alias("maximo")).orderBy("linea","hora_num"))
display(reales.groupBy("linea","tipo_dia","hora_num").agg(F.avg("demanda").alias("promedio"),F.count("demanda").alias("observaciones")).orderBy("linea","tipo_dia","hora_num"))
display(reales.groupBy("anio","mes","linea").agg(F.sum("demanda").alias("afluencia_observada"),F.count("demanda").alias("horas_observadas")).orderBy("anio","mes","linea"))
display(gold.filter("en_horario_operacion = 0 AND demanda > 0").groupBy("linea","hora_num").count())

## 5. Diagnóstico de faltantes

In [ ]:
faltantes_base=base.filter('demanda_original IS NULL AND en_horario_operacion = 1')
display(base.groupBy('linea','anio','mes','estado_dato_original','en_horario_operacion').count())
display(faltantes_base.groupBy('linea','hora_num').count())
calor(base.withColumn('faltante',((F.col('demanda_original').isNull()) & (F.col('en_horario_operacion')==1)).cast('int')),'faltante','Proporción de faltantes')


## 6. Evaluación temporal de imputación
Julio–septiembre de 2025 constituye un bloque de validación oculto. Las ventanas se calculan desde el corte de entrenamiento. Selección por MAE; antes de octubre se usa mediana histórica.

In [ ]:
VARIABLES = ["hora_num","dia_semana_num","mes","es_festivo","es_dia_previo_festivo","es_dia_posterior_festivo","operacion_parcial"]
NIVELES = [["linea","hora_num","dia_semana_num","es_festivo"],["linea","hora_num","dia_semana_num"],["linea","hora_num"],["linea"]]

def ajustar(algoritmo, datos, params=None):
    assert datos.count() > 0
    if algoritmo == "Mediana histórica":
        tablas = [datos.groupBy(*cols).agg(F.expr("percentile(demanda, 0.5)").alias(f"mediana_{i}")) for i,cols in enumerate(NIVELES)]
        global_ = datos.agg(F.expr("percentile(demanda, 0.5)")).first()[0]
        return tablas, global_
    indexador = StringIndexer(inputCol="linea",outputCol="linea_idx",handleInvalid="keep",stringOrderType="alphabetAsc")
    encoder = OneHotEncoder(inputCols=["linea_idx"],outputCols=["linea_vec"],dropLast=False)
    assembler = VectorAssembler(inputCols=["linea_vec",*VARIABLES],outputCol="features")
    rf = RandomForestRegressor(labelCol="demanda",predictionCol="prediccion",numTrees=(params or RF_PARAMS)[0],maxDepth=(params or RF_PARAMS)[1],minInstancesPerNode=(params or RF_PARAMS)[2],seed=42)
    return Pipeline(stages=[indexador,encoder,assembler,rf]).fit(datos)

def predecir(algoritmo, modelo, datos):
    if algoritmo == "Mediana histórica":
        tablas, global_ = modelo
        for cols,tabla in zip(NIVELES,tablas):
            datos = datos.join(F.broadcast(tabla),cols,"left")
        nombres = [f"mediana_{i}" for i in range(len(NIVELES))]
        return datos.withColumn("prediccion",F.coalesce(*[F.col(c) for c in nombres],F.lit(global_))).drop(*nombres)
    return modelo.transform(datos).drop("features","linea_idx","linea_vec")

def metricas(df, grupos=()):
    assert df.filter("prediccion IS NULL OR isnan(prediccion) OR prediccion < 0").count() == 0
    error = F.col("demanda")-F.col("prediccion")
    return (df.groupBy(*grupos).agg(F.count("*").alias("registros"),F.avg(F.abs(error)).alias("MAE"),F.sqrt(F.avg(error*error)).alias("RMSE"),F.sum(F.abs(error)).alias("sae"),F.sum(F.abs("demanda")).alias("den"),F.sum(error*error).alias("sse"),F.var_pop("demanda").alias("var"))
        .withColumn("WAPE",F.when(F.col("den")>0,100*F.col("sae")/F.col("den")))
        .withColumn("R2",F.when(F.col("var")>0,1-F.col("sse")/(F.col("var")*F.col("registros"))))
        .drop("sae","den","sse","var"))


ALGORITMOS=['Mediana histórica','Random Forest']
METODOS=['Mediana histórica','Mediana móvil 4','Mediana móvil 8','Mediana móvil 12','Random Forest']
def estimar_imputacion(metodo, referencia, destino, corte, params=None):
    if metodo.startswith('Mediana móvil'):
        semanas=int(metodo.split()[-1]); reciente=referencia.filter(F.col('fecha')>=F.lit(corte-timedelta(weeks=semanas)))
        if reciente.take(1):
            # Si una línea no tiene historia reciente, usar la referencia histórica.
            disponibles=reciente.select('linea').distinct()
            a=predecir('Mediana histórica',ajustar('Mediana histórica',reciente),destino.join(disponibles,'linea','inner'))
            resto=destino.join(disponibles,'linea','left_anti')
            return a.unionByName(predecir('Mediana histórica',ajustar('Mediana histórica',referencia),resto))
        metodo='Mediana histórica'
    return predecir(metodo,ajustar(metodo,referencia,params),destino)
imp_train=reales.filter(F.col('fecha')<F.lit(date(2025,7,1)))
imp_test=reales.filter(F.col('fecha').between('2025-07-01','2025-09-30'))
filas=[]
for metodo in METODOS:
    for params in (RF_GRID if metodo=='Random Forest' else [None]):
        pred=estimar_imputacion(metodo,imp_train,imp_test,date(2025,7,1),params)
        for r in metricas(pred,['linea']).collect():
            filas.append((r.linea,metodo,','.join(map(str,params)) if params else '',r.MAE,r.RMSE,r.WAPE))
comparacion_imp=spark.createDataFrame(filas,'linea string, metodo string, parametros string, MAE double, RMSE double, WAPE double')
metodos=comparacion_imp.withColumn('orden',F.row_number().over(Window.partitionBy('linea').orderBy('MAE','metodo','parametros'))).filter('orden=1').select('linea','metodo','parametros')
display(comparacion_imp); display(metodos)


## 7. Reconstrucción histórica
Cortes mensuales conservadores: ningún dato del mes reconstruido participa en el ajuste.

In [ ]:
faltantes=faltantes_base.join(metodos,'linea')
partes=[]
for r in faltantes.select(F.trunc('fecha','month').alias('mes_corte')).distinct().orderBy('mes_corte').collect():
    corte=r.mes_corte
    referencia=reales.filter(F.col('fecha')<F.lit(corte))
    pendientes=faltantes.filter(F.trunc('fecha','month')==F.lit(corte)).join(referencia.select('linea').distinct(),'linea','inner')
    if corte<date(2025,10,1): pendientes=pendientes.withColumn('metodo',F.lit('Mediana histórica')).withColumn('parametros',F.lit(''))
    for config in pendientes.select('metodo','parametros').distinct().collect():
        destino=pendientes.filter((F.col('metodo')==config.metodo)&(F.col('parametros')==config.parametros))
        params=tuple(map(int,config.parametros.split(','))) if config.parametros else None
        pred=estimar_imputacion(config.metodo,referencia,destino,corte,params)
        parte=pred.select(*CLAVES,F.round('prediccion').alias('estimacion'),F.col('metodo').alias('metodo_imputacion'),F.date_sub(F.lit(corte),1).alias('corte_imputacion'))
        partes.extend(parte.collect())
imputaciones=spark.createDataFrame(partes,'fecha date, linea string, hora_num int, estimacion double, metodo_imputacion string, corte_imputacion date') if partes else base.limit(0).select(*CLAVES,F.lit(None).cast('double').alias('estimacion'),F.lit(None).cast('string').alias('metodo_imputacion'),F.lit(None).cast('date').alias('corte_imputacion'))
base = base.join(imputaciones,CLAVES,"left")
base = (base.withColumn("demanda",F.coalesce("demanda_original","estimacion",F.when(F.col("en_horario_operacion")==0,0.0)))
    .withColumn("es_imputado",F.col("estimacion").isNotNull().cast("int"))
    .withColumn("origen_demanda",F.when(F.col("demanda_original").isNotNull(),F.when(F.col("en_horario_operacion")==1,"OBSERVADO").otherwise("OBSERVADO_FUERA_HORARIO")).when(F.col("es_imputado")==1,"IMPUTADO").when(F.col("en_horario_operacion")==0,"CERO_ESTRUCTURAL").otherwise("SIN_HISTORIA"))
    .fillna("NO_APLICA",subset=["metodo_imputacion"])
    .withColumn("total_general",F.sum("demanda").over(Window.partitionBy("fecha","linea"))).drop("estimacion","fila_presente"))
verificar(base,CLAVES)
assert base.count()==57700
assert base.filter("isnan(demanda) OR demanda < 0").count()==0
assert base.filter("demanda IS NULL AND origen_demanda != 'SIN_HISTORIA'").count()==0
assert base.filter("es_imputado = 1 AND (corte_imputacion IS NULL OR corte_imputacion >= fecha)").count()==0
assert base.filter("demanda_original IS NOT NULL AND demanda != demanda_original").count()==0
base = guardar(base,TABLA_GOLD_COMPLETA)
display(base.groupBy("anio","linea","origen_demanda").count().orderBy("anio","linea","origen_demanda"))

## 8. Demanda histórica consolidada

In [ ]:
display(base.groupBy('linea','origen_demanda').count())
perfil(base.filter('en_horario_operacion=1'),'demanda','Perfil reconstruido')
p=base.groupBy('linea','fecha').agg(F.sum('demanda_original').alias('observada'),F.sum('demanda').alias('reconstruida')).toPandas()
for linea,g in p.groupby('linea'):
    g=g.sort_values('fecha'); ax=g.plot(x='fecha',y=['observada','reconstruida'],figsize=(11,3),title=linea); ax.set_ylabel('Pasajeros'); plt.show(); plt.close()


## 9. Preparación para modelación

In [ ]:
datos = base.filter("en_horario_operacion = 1 AND demanda IS NOT NULL")
train = datos.filter(F.col("fecha").between("2025-01-01","2026-05-31"))
valid = datos.filter(F.col("fecha").between("2026-06-01","2026-06-30")).filter("origen_demanda = 'OBSERVADO'")
test = datos.filter(F.col("fecha").between("2026-07-01","2026-07-31")).filter("origen_demanda = 'OBSERVADO'")
for conjunto in [train,valid,test]:
    assert conjunto.count()>0 and conjunto.select("linea").distinct().count()==5

def seleccionar(df, escenario):
    return df.filter("origen_demanda = 'OBSERVADO'") if escenario == "Observados" else df

mlflow.set_experiment(EXPERIMENTO_MLFLOW)
mlflow.autolog(disable=True)

def registrar_metricas(df):
    fila = metricas(df).first().asDict()
    mlflow.log_metrics({k:float(v) for k,v in fila.items() if v is not None})
    return fila



## 10. Entrenamiento y selección

In [ ]:
resultados=[]
for algoritmo,escenario in [('Mediana histórica','Observados'),('Random Forest','Observados'),('Random Forest','Observados e imputados')]:
    entrenamiento=seleccionar(train,escenario)
    for params in (RF_GRID if algoritmo=='Random Forest' else [None]):
        etiqueta=','.join(map(str,params)) if params else ''
        with mlflow.start_run(run_name=f'{algoritmo} | {escenario} | {etiqueta}') as run:
            mlflow.log_params({'algoritmo':algoritmo,'escenario':escenario,'parametros':etiqueta,'corte':'2026-05-31','semilla':42})
            estimador=ajustar(algoritmo,entrenamiento,params)
            pred=predecir(algoritmo,estimador,valid)
            m=registrar_metricas(pred)
            resultados.append((algoritmo,escenario,etiqueta,run.info.run_id,m['MAE'],m['RMSE'],m['R2'],m['WAPE']))
comparacion=spark.createDataFrame(resultados,'algoritmo string, escenario string, parametros string, run_id string, MAE double, RMSE double, R2 double, WAPE double')
comparacion=guardar(comparacion,tabla_analitica('metrodemanda_comparacion_modelos'))
display(comparacion.orderBy('WAPE'))
ganador=comparacion.filter('WAPE IS NOT NULL').orderBy('WAPE','algoritmo','escenario','parametros').first()
algoritmo_ganador,escenario_ganador=ganador.algoritmo,ganador.escenario
if ganador.parametros: RF_PARAMS=tuple(map(int,ganador.parametros.split(',')))


## 11. Evaluación independiente

In [ ]:
prejulio = seleccionar(datos.filter(F.col("fecha").between("2025-01-01","2026-06-30")),escenario_ganador)
with mlflow.start_run(run_name="Prueba independiente julio"):
    mlflow.log_params({"algoritmo":algoritmo_ganador,"escenario":escenario_ganador,"corte":"2026-06-30"})
    modelo_prueba = ajustar(algoritmo_ganador,prejulio)
    evaluacion = predecir(algoritmo_ganador,modelo_prueba,test).select(*CLAVES,"demanda","prediccion")
    print(registrar_metricas(evaluacion))
    evaluacion = guardar(evaluacion,tabla_analitica("metrodemanda_evaluacion_julio2026"))
display(metricas(evaluacion,["linea"]).orderBy("linea"))
display(evaluacion.groupBy("fecha","linea").agg(F.sum("demanda").alias("observada"),F.sum("prediccion").alias("estimada")).orderBy("fecha","linea"))
display(metricas(evaluacion,['linea','hora_num']))
p=evaluacion.groupBy('linea','fecha').agg(F.sum('demanda').alias('real'),F.sum('prediccion').alias('predicha')).toPandas()
for linea,g in p.groupby('linea'):
    g.sort_values('fecha').plot(x='fecha',y=['real','predicha'],figsize=(11,3),title=f'Julio — {linea}'); plt.ylabel('Pasajeros en horas observadas'); plt.show(); plt.close()
p=evaluacion.groupBy('linea','hora_num').agg(F.avg('demanda').alias('real'),F.avg('prediccion').alias('predicha')).toPandas()
for linea,g in p.groupby('linea'):
    g.sort_values('hora_num').plot(x='hora_num',y=['real','predicha'],figsize=(9,3),title=f'Perfil horario — {linea}'); plt.ylabel('Pasajeros'); plt.show(); plt.close()
p=evaluacion.select('linea',(F.col('demanda')-F.col('prediccion')).alias('residuo')).toPandas()
for linea,g in p.groupby('linea'):
    plt.hist(g.residuo,bins=30); plt.title(f'Residuos — {linea}'); plt.show(); plt.close()


## 12. Modelo definitivo y persistencia

In [ ]:
from mlflow.models import infer_signature
os.environ["MLFLOW_DFS_TMP"] = RUTA_DATA.rstrip("/")
historico = seleccionar(datos.filter(F.col("fecha").between("2025-01-01","2026-07-31")),escenario_ganador)
with mlflow.start_run(run_name="Modelo definitivo") as run:
    run_definitivo = run.info.run_id
    mlflow.log_params({"algoritmo":algoritmo_ganador,"escenario":escenario_ganador,"corte":"2026-07-31","registros":historico.count()})
    modelo_definitivo = ajustar(algoritmo_ganador,historico)
    if algoritmo_ganador == "Random Forest":
        try:
            muestra = historico.select("linea",*VARIABLES).limit(5)
            firma = infer_signature(muestra,modelo_definitivo.transform(muestra).select("prediccion"))
            artefacto = mlflow.spark.log_model(modelo_definitivo,artifact_path="modelo",signature=firma,dfs_tmpdir=RUTA_DATA.rstrip("/"))
            recuperado = mlflow.spark.load_model(artefacto.model_uri,dfs_tmpdir=RUTA_DATA.rstrip("/"))
            mlflow.set_tag("persistencia_modelo","MLflow")
            print(artefacto.model_uri)
        except Exception as error:
            print("MLflow no pudo guardar el artefacto:",str(error))
            ruta_modelo = RUTA_DATA + "metrodemanda_modelo_" + run_definitivo
            modelo_definitivo.write().overwrite().save(ruta_modelo)
            recuperado = PipelineModel.load(ruta_modelo)
            mlflow.set_tag("persistencia_modelo",ruta_modelo)
    else:
        tablas, global_ = modelo_definitivo
        for i,tabla in enumerate(tablas):
            guardar(tabla,tabla_analitica(f"metrodemanda_mediana_{run_definitivo}_nivel_{i}"))
        mlflow.log_param("mediana_global",global_)
        recuperado = ([spark.table(tabla_analitica(f"metrodemanda_mediana_{run_definitivo}_nivel_{i}")) for i in range(len(tablas))], float(mlflow.get_run(run_definitivo).data.params["mediana_global"]))
        mlflow.set_tag("persistencia_modelo","Tablas Delta de medianas")
    muestra_control = historico.select(*dict.fromkeys(CLAVES+VARIABLES)).orderBy(*CLAVES).limit(100)
    antes = predecir(algoritmo_ganador,modelo_definitivo,muestra_control).select(*CLAVES,F.col("prediccion").alias("antes"))
    despues = predecir(algoritmo_ganador,recuperado,muestra_control).select(*CLAVES,F.col("prediccion").alias("despues"))
    control = antes.join(despues,CLAVES,"full")
    assert control.filter("antes IS NULL OR despues IS NULL OR abs(antes-despues)>0.000001").count()==0
    mlflow.set_tag("recuperacion_verificada","true")
modelo_definitivo = recuperado
print("Modelo definitivo:",algoritmo_ganador,escenario_ganador,run_definitivo)

## 13. Pronóstico horario

In [ ]:
horizonte = estructura("2026-08-01","2026-12-31")
operativas = predecir(algoritmo_ganador,modelo_definitivo,horizonte.filter("en_horario_operacion = 1")).select(*CLAVES,"prediccion")
assert operativas.filter("prediccion IS NULL OR isnan(prediccion) OR prediccion < 0").count()==0
pronostico = (horizonte.join(operativas,CLAVES,"left")
    .withColumn("demanda_predicha",F.when(F.col("en_horario_operacion")==0,0).otherwise(F.round("prediccion")).cast("long"))
    .withColumn("tipo_ejercicio",F.lit("SIMULACION_FUERA_DE_MUESTRA"))
    .withColumn("run_id_modelo",F.lit(run_definitivo)).withColumn("fecha_generacion",F.current_timestamp()).drop("prediccion"))
verificar(pronostico,CLAVES)
assert pronostico.count()==15300
assert pronostico.filter("demanda_predicha IS NULL OR demanda_predicha < 0 OR (en_horario_operacion = 0 AND demanda_predicha != 0)").count()==0
pronostico = guardar(pronostico,TABLA_PREDICCIONES)
display(pronostico.groupBy("linea","mes").agg(F.sum("demanda_predicha").alias("afluencia_estimada")).orderBy("linea","mes"))

## 14. Análisis del pronóstico

In [ ]:
operativo = pronostico.filter("en_horario_operacion = 1")
display(operativo.groupBy("linea","hora_num").agg(F.avg("demanda_predicha").alias("promedio"),F.stddev("demanda_predicha").alias("desviacion"),F.min("demanda_predicha").alias("minimo"),F.max("demanda_predicha").alias("maximo")).orderBy("linea","hora_num"))
diario = pronostico.groupBy("fecha","linea","tipo_dia").agg(F.sum("demanda_predicha").alias("demanda_diaria"))
display(diario.groupBy("linea","tipo_dia").agg(F.count("*").alias("dias"),F.avg("demanda_diaria").alias("promedio"),F.stddev("demanda_diaria").alias("desviacion")))
display(operativo.orderBy("demanda_predicha").limit(10))
display(operativo.orderBy(F.desc("demanda_predicha")).limit(10))
print("Registros históricos:",base.count(),"| Pronósticos:",pronostico.count())
picos = operativo.groupBy("linea","tipo_dia","hora_num").agg(F.avg("demanda_predicha").alias("promedio"))
display(picos.withColumn("posicion",F.row_number().over(Window.partitionBy("linea","tipo_dia").orderBy(F.desc("promedio"),"hora_num"))).filter("posicion <= 3"))

perfil(pronostico,'demanda_predicha','Perfil pronosticado')
calor(pronostico,'demanda_predicha','Demanda pronosticada')
grafica_lineas(pronostico.groupBy('linea',F.trunc('fecha','month').alias('mes_fecha')).agg(F.sum('demanda_predicha').alias('total')),'mes_fecha','total','Pronóstico mensual')
p=base.groupBy('linea',F.trunc('fecha','month').alias('mes_fecha')).agg(F.sum('demanda').alias('historica')).toPandas()
q=pronostico.groupBy('linea',F.trunc('fecha','month').alias('mes_fecha')).agg(F.sum('demanda_predicha').alias('pronostico')).toPandas()
for linea,g in p.merge(q,on=['linea','mes_fecha'],how='outer').groupby('linea'):
    g.sort_values('mes_fecha').plot(x='mes_fecha',y=['historica','pronostico'],title=linea,figsize=(11,3)); plt.ylabel('Pasajeros'); plt.show(); plt.close()


## 15. Validación integral

In [ ]:
verificar(base,CLAVES); verificar(pronostico,CLAVES)
assert base.count()==57700 and pronostico.count()==15300
assert base.filter('demanda_original IS NOT NULL AND NOT (demanda <=> demanda_original)').count()==0
assert base.filter('es_imputado=1 AND (corte_imputacion IS NULL OR corte_imputacion>=fecha)').count()==0
assert pronostico.filter('demanda_predicha IS NULL OR demanda_predicha<0').count()==0
assert control.filter('antes IS NULL OR despues IS NULL OR abs(antes-despues)>0.000001').count()==0
print('Controles integrales superados')
